# Capítulo 3 — Redes Neurais Artificiais e Deep Learning

**Inteligência Artificial e Machine Learning Avançado para Defesa** — Prof. Cristiano Alves · Quantum Strategic AI

---

### 🎯 Objetivos do capítulo

Ao final deste notebook, você será capaz de:

1. **Explicar** o *perceptron*, o seu limite fundamental (o problema do **XOR**) e como o *perceptron multicamadas* (MLP) o supera pela composição de camadas com ativação não linear;
2. **Descrever** o *gradiente descendente* e o algoritmo de *retropropagação* como o mecanismo pelo qual uma rede aprende, e compreender o papel das **funções de ativação** e o problema do **gradiente evanescente**;
3. **Distinguir** as principais arquiteturas profundas — *redes convolucionais* (CNN), *recorrentes* (RNN) e *LSTM* — e o tipo de dado a que cada uma se adequa;
4. **Implementar e treinar** redes em `TensorFlow/Keras` e em `PyTorch`, reconhecendo os idiomas **declarativo** e **imperativo** de cada biblioteca;
5. **Aplicar** *regularização* (*dropout*, *weight decay*, *batch normalization*), otimizadores modernos e boas práticas de treinamento para controlar o sobreajuste em problemas operacionais.

> O Capítulo 2 percorreu o arsenal clássico do aprendizado supervisionado e, mais importante que qualquer modelo isolado, firmou uma disciplina: comparar modelos honestamente contra uma *baseline*, sob validação cruzada, com métricas ajustadas ao custo operacional do erro. Neste capítulo damos o salto para as **redes neurais profundas** — a família de modelos que, na última década, redefiniu o que é possível em percepção (visão e linguagem) e que constitui o motor dos Módulos II e III. Com ele, fechamos o Módulo I.
>
> Uma advertência abre o capítulo, e é a mesma que fechou o anterior: **profundidade não é sofisticação gratuita**. Uma rede profunda só se justifica, em um sistema crítico, quando supera com folga mensurável a *baseline* clássica — e o faz ao custo de menor auditabilidade. O aumento de capacidade dos modelos **não dissolve a disciplina do Capítulo 2; torna-a mais necessária**.

## Preparação do ambiente

O Google Colab já traz instaladas todas as bibliotecas deste capítulo (`numpy`, `matplotlib`, `scikit-learn`, `tensorflow`, `torch`). Todas as células executam em CPU em poucos minutos; se quiser acelerar os treinos das CNNs, ative uma GPU gratuita em *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)*.

A célula abaixo fixa as versões usadas e as **sementes aleatórias** — requisito de reprodutibilidade que, em sistemas de defesa, não é opcional: *dados os mesmos dados, os mesmos resultados*. Em redes neurais há **três** fontes de aleatoriedade a controlar: a geração dos dados (`numpy`), a inicialização dos pesos e o embaralhamento dos minilotes (`tensorflow` e `torch`).

In [ ]:
# Preparacao do ambiente: importacoes e sementes de reprodutibilidade
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import torch
import torch.nn as nn

SEMENTE = 0
np.random.seed(SEMENTE)
keras.utils.set_random_seed(SEMENTE)   # python, numpy e tensorflow
torch.manual_seed(SEMENTE)

print(f"numpy        {np.__version__}")
print(f"scikit-learn {sklearn.__version__}")
print(f"tensorflow   {tf.__version__}")
print(f"pytorch      {torch.__version__}")
print("GPU disponivel (TF):", bool(tf.config.list_physical_devices("GPU")))
print("Ambiente pronto.")

---
## 3.1 Do perceptron ao perceptron multicamadas

A **tradição conexionista**, apresentada no Capítulo 1, nasce de uma metáfora biológica: se o cérebro computa por meio de neurônios que se excitam e se inibem, talvez se possa aprender construindo unidades de cálculo análogas. O modelo de McCulloch e Pitts (1943) e o *perceptron* de Rosenblatt (1958) são as sementes dessa ideia — e o ponto de partida natural para entender o *deep learning*.

### 3.1.1 O perceptron

O perceptron é a unidade elementar. Dado um vetor de entrada $\mathbf{x} = (x_1, \ldots, x_d)$, um vetor de pesos $\mathbf{w}$ e um viés $b$, ele calcula uma combinação linear e a submete a uma **função de ativação em degrau**:

$$\hat{y} = \varphi\left(\mathbf{w}^\top \mathbf{x} + b\right), \qquad \varphi(z) = \begin{cases} 1, & z \geq 0, \\ 0, & z < 0. \end{cases}$$

Geometricamente, $\mathbf{w}^\top \mathbf{x} + b = 0$ é um **hiperplano** que divide o espaço de entrada em dois semiespaços; o perceptron classifica conforme o lado. O aprendizado ajusta os pesos por uma regra simples e **local**: a cada exemplo mal classificado,

$$\mathbf{w} \leftarrow \mathbf{w} + \eta\,(y - \hat{y})\,\mathbf{x},$$

onde $\eta$ é a *taxa de aprendizado*. O **teorema de convergência do perceptron** garante que, se os dados forem *linearmente separáveis*, esse processo termina em um número finito de passos.

O leitor reconhecerá aqui o **ancestral da regressão logística** do Capítulo 2 — a diferença é que a logística substitui o degrau por uma sigmoide suave e devolve probabilidades, o que a torna treinável por gradiente e muito mais informativa.

A célula abaixo implementa o perceptron **do zero**, em poucas linhas, sobre um problema linearmente separável de dois atributos de sensor:

In [ ]:
# O perceptron de Rosenblatt implementado do zero
rng = np.random.default_rng(SEMENTE)

# Problema linearmente separavel: dois grupos de contatos descritos por
# dois atributos de sensor (ex.: intensidade do eco, largura do pulso)
n = 120
X_neu = rng.normal([-1.5, -1.0], 0.6, (n // 2, 2))
X_hos = rng.normal([ 1.5,  1.0], 0.6, (n // 2, 2))
X = np.vstack([X_neu, X_hos])
y = np.array([0] * (n // 2) + [1] * (n // 2))

def treina_perceptron(X, y, eta=0.1, max_epocas=50):
    w, b = np.zeros(X.shape[1]), 0.0
    erros_por_epoca = []
    for epoca in range(max_epocas):
        erros = 0
        for xi, yi in zip(X, y):
            y_prev = 1 if xi @ w + b >= 0 else 0
            if y_prev != yi:                 # regra local do perceptron
                w += eta * (yi - y_prev) * xi
                b += eta * (yi - y_prev)
                erros += 1
        erros_por_epoca.append(erros)
        if erros == 0:                       # convergiu: separou tudo
            break
    return w, b, erros_por_epoca

w, b, historico = treina_perceptron(X, y)
print(f"Convergiu em {len(historico)} epocas | "
      f"w = {np.round(w, 3)}, b = {b:.3f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
a1.scatter(*X[y == 0].T, c="tab:blue", label="neutro", alpha=0.7)
a1.scatter(*X[y == 1].T, c="tab:red", marker="^", label="hostil", alpha=0.7)
xs = np.linspace(X[:, 0].min(), X[:, 0].max(), 2)
a1.plot(xs, -(w[0] * xs + b) / w[1], "k--", lw=2, label="hiperplano aprendido")
a1.set_xlabel("atributo 1"); a1.set_ylabel("atributo 2")
a1.set_title("Fronteira de decisao do perceptron"); a1.legend()
a2.plot(range(1, len(historico) + 1), historico, "o-")
a2.set_xlabel("epoca"); a2.set_ylabel("exemplos mal classificados")
a2.set_title("Convergencia (teorema do perceptron)")
plt.tight_layout(); plt.show()

**Observe:** o número de erros por época cai até **zero** — o teorema de convergência em ação. Mas a garantia vale *apenas* para dados linearmente separáveis. E é exatamente aí que a história muda de rumo.

### 3.1.2 O limite do perceptron: o problema do XOR

A limitação é séria e histórica. Minsky e Papert, em *Perceptrons* (1969), demonstraram que um único perceptron **não representa a função lógica XOR** (ou-exclusivo): não existe reta que separe os pares $\{(0,0), (1,1)\}$ de $\{(0,1), (1,0)\}$, porque o problema **não é linearmente separável**. A repercussão desse resultado é frequentemente apontada como um dos gatilhos do primeiro "inverno da IA" discutido no Capítulo 1: o financiamento à abordagem conexionista minguou por mais de uma década.

A célula seguinte submete o *mesmo* perceptron ao XOR — e o vê falhar:

In [ ]:
# O problema do XOR: o perceptron falha onde nao ha reta separadora
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])   # ou-exclusivo

w, b, historico = treina_perceptron(X_xor, y_xor, max_epocas=200)
print(f"Epocas executadas: {len(historico)} (nunca converge)")
print(f"Erros nas ultimas 10 epocas: {historico[-10:]}")

plt.figure(figsize=(5, 4.5))
for classe, cor, marcador in [(0, "tab:blue", "o"), (1, "tab:red", "^")]:
    pts = X_xor[y_xor == classe]
    plt.scatter(*pts.T, c=cor, marker=marcador, s=180,
                label=f"classe {classe}")
for (px, py), rotulo in zip(X_xor, y_xor):
    plt.annotate(f"({px:.0f}, {py:.0f})", (px, py),
                 textcoords="offset points", xytext=(10, 8))
plt.xlim(-0.5, 1.5); plt.ylim(-0.5, 1.5)
plt.title("XOR: nenhuma reta separa as duas classes")
plt.legend(); plt.tight_layout(); plt.show()

**Observe:** os erros **oscilam indefinidamente** — a regra de aprendizado não tem como terminar, porque a reta que ela procura *não existe*.

A saída — conhecida em princípio, mas só viável na prática com a retropropagação — é **empilhar** unidades em camadas, intercalando uma **não linearidade**. Uma camada intermediária constrói *representações novas* do dado, em que o problema, antes emaranhado, torna-se separável. É esse o salto conceitual que funda o aprendizado profundo.

### 3.1.3 O perceptron multicamadas (MLP)

O *perceptron multicamadas* organiza neurônios em camadas sucessivas. Com uma camada oculta, a rede calcula

$$\mathbf{h} = \varphi\left(\mathbf{W}^{(1)} \mathbf{x} + \mathbf{b}^{(1)}\right), \qquad \hat{y} = \psi\left(\mathbf{W}^{(2)} \mathbf{h} + \mathbf{b}^{(2)}\right),$$

em que $\varphi$ é uma ativação não linear (tipicamente a **ReLU**, adiante) e $\psi$ é a ativação de saída — sigmoide para decisão binária, *softmax* para classificação multiclasse. As camadas intermediárias não são observadas diretamente: são **representações latentes** que a rede aprende para tornar a tarefa solúvel.

O poder dessa construção é formalizado pelo **teorema da aproximação universal** (Cybenko, 1989; Hornik, 1991): uma rede com uma única camada oculta e número suficiente de unidades aproxima qualquer função contínua sobre um conjunto compacto, com a precisão que se queira. O teorema é de **existência, não de eficiência** — na prática, *profundidade* (muitas camadas) costuma ser mais econômica em parâmetros do que *largura* (uma camada enorme), e é a profundidade que dá nome ao *deep learning*.

> **🛡️ Contexto de defesa** — Em defesa, o MLP é a ferramenta natural para a **fusão de dados tabulares** de múltiplos sensores — combinar leituras heterogêneas (radar, acústica, sinais) em uma decisão única. Para dados com estrutura espacial (imagens) ou temporal (séries, trajetórias), porém, **arquiteturas especializadas**, tratadas na Seção 3.3, exploram essa estrutura e superam largamente um MLP genérico — tanto em desempenho quanto em economia de parâmetros, o que importa diretamente para a IA embarcada do Módulo IV.

A célula abaixo confronta, sobre uma **nuvem XOR ruidosa**, um modelo linear (a regressão logística do Capítulo 2) e um MLP com uma única camada oculta:

In [ ]:
# Regressao logistica (linear) vs. MLP em uma nuvem XOR ruidosa
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(SEMENTE)
n = 400
centros = np.array([[-1, -1], [-1, 1], [1, -1], [1, 1]], dtype=float)
rotulos_centros = np.array([0, 1, 1, 0])
indices = rng.integers(0, 4, n)
X = centros[indices] + rng.normal(0, 0.35, (n, 2))
y = rotulos_centros[indices]

logistica = LogisticRegression().fit(X, y)

# MLP minimo: UMA camada oculta com 8 unidades tanh
mlp = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(8, activation="tanh"),
    layers.Dense(1, activation="sigmoid"),
])
mlp.compile(optimizer="adam", loss="binary_crossentropy")
mlp.fit(X, y, epochs=300, batch_size=32, verbose=0)

# Superficies de decisao lado a lado
gx, gy = np.meshgrid(np.linspace(-2.4, 2.4, 200),
                     np.linspace(-2.4, 2.4, 200))
grade = np.column_stack([gx.ravel(), gy.ravel()])
Z_log = logistica.predict_proba(grade)[:, 1].reshape(gx.shape)
Z_mlp = mlp.predict(grade, verbose=0).reshape(gx.shape)

ac_log = accuracy_score(y, logistica.predict(X))
ac_mlp = accuracy_score(y, mlp.predict(X, verbose=0).ravel() >= 0.5)

fig, eixos = plt.subplots(1, 2, figsize=(11, 4.5))
for eixo, Z, titulo, ac in [
        (eixos[0], Z_log, "Regressao logistica (linear)", ac_log),
        (eixos[1], Z_mlp, "MLP com 1 camada oculta", ac_mlp)]:
    eixo.contourf(gx, gy, Z, levels=20, cmap="RdBu_r", alpha=0.75)
    eixo.scatter(*X[y == 0].T, c="tab:blue", s=12)
    eixo.scatter(*X[y == 1].T, c="tab:red", s=12, marker="^")
    eixo.set_title(f"{titulo} — acuracia {ac:.2f}")
plt.tight_layout(); plt.show()

**Observe:** o modelo linear estaciona perto da **acurácia do acaso** (~0,5) — sua fronteira é uma reta, e reta nenhuma resolve o XOR. O MLP, com **uma única camada oculta de 8 unidades**, desenha uma fronteira curva e resolve o problema: a camada oculta aprendeu uma *representação latente* em que as classes se separam.

Resta a pergunta central: **como** os pesos dessas camadas são encontrados? A resposta é o assunto da próxima seção — e o mecanismo que tornou o aprendizado profundo possível.

---
## 3.2 Retropropagação e o treinamento por gradiente

Uma rede só é útil depois de treinada. Treinar significa ajustar todos os pesos $\theta = \{\mathbf{W}^{(l)}, \mathbf{b}^{(l)}\}$ de modo a minimizar uma **função de perda** que meça o erro das previsões. O mecanismo que torna isso possível em redes profundas é a *retropropagação*.

### 3.2.1 A função de perda e o gradiente descendente

Seja $\mathcal{L}(\theta)$ a perda média sobre os dados de treino — **entropia cruzada** para classificação, **erro quadrático** para regressão (Capítulo 2). O **gradiente descendente** atualiza os parâmetros na direção que mais reduz a perda:

$$\theta \leftarrow \theta - \eta\, \nabla_\theta\, \mathcal{L}(\theta),$$

onde $\nabla_\theta \mathcal{L}$ é o vetor de derivadas parciais da perda em relação a cada parâmetro e $\eta$ é a **taxa de aprendizado**. Calcular o gradiente sobre *todo* o conjunto a cada passo é caro; na prática usa-se o **gradiente descendente estocástico em minilotes** (*mini-batch SGD*), que estima o gradiente sobre pequenos subconjuntos. Além de barato, o ruído dessa estimativa ajuda a escapar de mínimos rasos.

A célula abaixo visualiza o gradiente descendente sobre uma "tigela" de perda alongada — e mostra por que a **taxa de aprendizado** é o hiperparâmetro mais crítico do treinamento:

In [ ]:
# Gradiente descendente em uma "tigela" de perda alongada:
# o efeito da taxa de aprendizado eta
def gradiente(w):
    return np.array([w[0], 10 * w[1]])   # grad de 0.5*(w1^2 + 10*w2^2)

gx, gy = np.meshgrid(np.linspace(-10, 10, 200), np.linspace(-6, 6, 200))
Z = 0.5 * (gx**2 + 10 * gy**2)

plt.figure(figsize=(8.5, 5))
plt.contour(gx, gy, Z, levels=25, cmap="Greys", alpha=0.6)
for eta, cor, passos, rotulo in [
        (0.02, "tab:blue",  60, "lenta demais"),
        (0.09, "tab:green", 60, "adequada"),
        (0.21, "tab:red",    9, "alta demais: DIVERGE")]:
    w = np.array([-9.0, 4.5])
    caminho = [w.copy()]
    for _ in range(passos):
        w = w - eta * gradiente(w)
        caminho.append(w.copy())
    caminho = np.array(caminho)
    plt.plot(caminho[:, 0], caminho[:, 1], "o-", ms=3, color=cor,
             label=f"eta = {eta} ({rotulo})")
plt.plot(0, 0, "k*", ms=16, label="minimo da perda")
plt.xlim(-10, 10); plt.ylim(-6, 6)
plt.xlabel("$w_1$"); plt.ylabel("$w_2$")
plt.title("Tres taxas de aprendizado, tres destinos")
plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

**Observe:** com $\eta$ pequeno o treino **se arrasta**; com $\eta$ adequado, converge; **alto demais, diverge** — os passos saltam sobre o mínimo e a perda explode. Guarde esta figura: ela reaparecerá na Seção 3.4, quando discutirmos otimizadores.

### 3.2.2 O algoritmo de retropropagação

O obstáculo é calcular $\nabla_\theta \mathcal{L}$ para uma rede com muitas camadas. A **retropropagação** resolve isso aplicando a *regra da cadeia* de forma organizada, propagando o erro da saída para as entradas. O procedimento tem dois passos. No **passo para frente**, calculam-se as ativações de cada camada até a saída e a perda. No **passo para trás**, calcula-se um sinal de erro $\boldsymbol{\delta}$ na saída e propaga-se para trás:

$$\boldsymbol{\delta}^{(l)} = \left(\mathbf{W}^{(l+1)}\right)^\top \boldsymbol{\delta}^{(l+1)} \odot \varphi'\left(\mathbf{z}^{(l)}\right), \qquad \frac{\partial \mathcal{L}}{\partial \mathbf{W}^{(l)}} = \boldsymbol{\delta}^{(l)} \left(\mathbf{a}^{(l-1)}\right)^\top,$$

onde $\mathbf{z}^{(l)}$ é a pré-ativação da camada $l$, $\mathbf{a}^{(l-1)}$ é a saída da camada anterior e $\odot$ é o produto elemento a elemento. Não é preciso decorar essas fórmulas: as bibliotecas as executam automaticamente por **diferenciação automática**. Mas entendê-las explica os modos de falha — em particular, por que gradientes podem "desaparecer" ao atravessar muitas camadas: a cada camada, o sinal de erro é **multiplicado** pela derivada da ativação.

### 3.2.3 Funções de ativação

A não linearidade $\varphi$ é indispensável: empilhar camadas puramente lineares **colapsa em um único mapa linear**, sem ganho de expressividade. As escolhas clássicas são três:

- **Sigmoide**, $\sigma(z) = 1/(1 + e^{-z})$: comprime a $[0, 1]$, mas **satura** — sua derivada não passa de 0,25 e tende a zero nos extremos. Em redes profundas, isso provoca o **gradiente evanescente**: o sinal de erro encolhe a cada camada e as primeiras camadas praticamente não aprendem.
- **Tangente hiperbólica**, $\tanh(z)$: semelhante à sigmoide, porém centrada em zero, o que ajuda a otimização; ainda satura.
- **ReLU**, $\varphi(z) = \max(0, z)$: derivada igual a 1 para $z > 0$, o que **preserva** o gradiente e acelera o treino; é barata e induz ativações esparsas. Tornou-se **o padrão nas camadas ocultas**. Seu risco — a "ReLU morta", unidade travada em zero — é mitigado por variantes como a *Leaky ReLU*.

Na **saída**, a ativação segue a tarefa: sigmoide para decisão binária, *softmax* para multiclasse.

In [ ]:
# Funcoes de ativacao, suas derivadas e o gradiente evanescente
z = np.linspace(-5, 5, 400)
sig = 1 / (1 + np.exp(-z))
ativacoes = {
    "Sigmoide": (sig, sig * (1 - sig)),
    "Tanh":     (np.tanh(z), 1 - np.tanh(z)**2),
    "ReLU":     (np.maximum(0, z), (z > 0).astype(float)),
}

fig, eixos = plt.subplots(2, 3, figsize=(12, 5.6), sharex=True)
for j, (nome, (f, df)) in enumerate(ativacoes.items()):
    eixos[0, j].plot(z, f, lw=2)
    eixos[0, j].set_title(nome)
    eixos[1, j].plot(z, df, lw=2, color="tab:orange")
    eixos[1, j].set_title(f"derivada (maximo = {df.max():.2f})")
    eixos[1, j].set_xlabel("z")
plt.tight_layout(); plt.show()

# O gradiente evanescente em numeros: o sinal de erro atravessa L
# camadas sendo MULTIPLICADO pela derivada em cada uma (melhor caso)
L = np.arange(1, 21)
plt.figure(figsize=(7, 4.2))
plt.semilogy(L, 0.25**L, "o-", label="sigmoide (fator 0.25 por camada)")
plt.semilogy(L, 1.0**L, "s-", label="ReLU (fator 1.0 por camada)")
plt.xlabel("numero de camadas atravessadas")
plt.ylabel("fator multiplicativo do gradiente (log)")
plt.title("Por que a ReLU substituiu a sigmoide nas camadas ocultas")
plt.legend(); plt.tight_layout(); plt.show()

print(f"Apos 10 camadas sigmoides (melhor caso), o gradiente "
      f"e reduzido por um fator de {0.25**10:.1e}")

**Observe:** no *melhor* caso, dez camadas sigmoides reduzem o gradiente em um fator de $10^{-6}$ — as primeiras camadas ficam, na prática, congeladas. A ReLU preserva o sinal e é uma das razões concretas pelas quais redes profundas se tornaram treináveis.

Com os fundamentos no lugar, a **Listagem 3.1** treina um MLP sobre o mesmo problema de **triagem de ameaças** do Capítulo 2 — porém com uma diferença deliberada: a classe positiva depende de forma **não linear** dos atributos (um produto entre atributos e um seno), terreno onde uma rede pode superar a *baseline*. Note a reutilização da disciplina do Capítulo 2: separação estratificada, padronização ajustada **só no treino** e **pesos de classe** para compensar o desbalanceamento.

In [ ]:
# Listagem 3.1 - Perceptron multicamadas em Keras para triagem de ameacas.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Dados sinteticos de triagem: n amostras, d atributos de sensor.
# A classe positiva (ameaca) e rara (~18%) e depende de forma NAO
# LINEAR dos atributos (interacao x0*x3 e um seno) -- terreno onde
# uma rede pode superar a baseline linear.
rng = np.random.default_rng(SEMENTE)
n, d = 4000, 12
X = rng.normal(size=(n, d))
logit = (3.0 * X[:, 0] * X[:, 3] - 1.0 * X[:, 5]
         + 1.5 * np.sin(X[:, 7]) - 3.0)
prob = 1.0 / (1.0 + np.exp(-logit))
y = (rng.uniform(size=n) < prob).astype("int32")

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, stratify=y, test_size=0.25, random_state=SEMENTE)

# Padronizacao ajustada SO no treino (evita vazamento -- ver Cap. 2).
scaler = StandardScaler().fit(X_tr)
X_tr, X_te = scaler.transform(X_tr), scaler.transform(X_te)

# MLP: duas camadas ocultas com ReLU; saida sigmoide (probabilidade).
model = keras.Sequential([
    layers.Input(shape=(d,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[keras.metrics.Recall(name="revocacao"),
             keras.metrics.Precision(name="precisao")])

# Pesos de classe compensam o desbalanceamento (analogo ao Cap. 2).
n_pos = max(int((y_tr == 1).sum()), 1)
peso = {0: 1.0, 1: float((y_tr == 0).sum() / n_pos)}

model.fit(X_tr, y_tr, validation_split=0.2, epochs=30,
          batch_size=64, class_weight=peso, verbose=0)

_, rev, prec = model.evaluate(X_te, y_te, verbose=0)
print(f"Revocacao (ameaca): {rev:.3f}   Precisao: {prec:.3f}")

**Observe:** o método `fit` cuidou de tudo — minilotes, embaralhamento, retropropagação, validação. É o idioma **declarativo**: descreve-se a arquitetura em alto nível e a biblioteca executa o treino.

> **📝 Nota** — As duas grandes bibliotecas de *deep learning* adotam idiomas distintos. `TensorFlow/Keras` é **declarativo**: descreve-se a arquitetura em alto nível e o método `fit` cuida do laço de treino — rápido para prototipar e implantar. `PyTorch` é **imperativo** (*define-by-run*): o grafo é construído a cada execução, em Python puro, o que dá controle fino e favorece a pesquisa. Ambos são padrão de mercado; a proposta desta obra emprega os dois. A **Listagem 3.2** reescreve a mesma rede em PyTorch, tornando a retropropagação **visível** no código.

In [ ]:
# Listagem 3.2 - A mesma rede em PyTorch: o laco de treino explicito.
# Reaproveita X_tr, y_tr ja padronizados, convertidos em tensores.
Xt = torch.tensor(X_tr, dtype=torch.float32)
yt = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1)

# Mesmo MLP no idioma imperativo do PyTorch.
rede = nn.Sequential(
    nn.Linear(d, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 1),            # sem sigmoide: usamos BCEWithLogits
)

# pos_weight compensa o desbalanceamento (papel do class_weight acima)
criterio = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(peso[1]))
otimizador = torch.optim.Adam(rede.parameters(), lr=1e-3)

# Laco de treino explicito: a retropropagacao aparece em tres linhas.
# (300 passos full-batch ~ os 30 x 38 minilotes do fit do Keras)
for epoca in range(300):
    otimizador.zero_grad()           # 1) zera os gradientes acumulados
    logits = rede(Xt)                # 2) passo para frente
    perda = criterio(logits, yt)     #    calcula a perda
    perda.backward()                 # 3) retropropagacao (autograd)
    otimizador.step()                #    atualiza os pesos

print(f"Perda final de treino: {perda.item():.4f}")

# Avaliacao no teste, para comparar com a versao Keras
from sklearn.metrics import recall_score, precision_score
with torch.no_grad():
    p_teste = torch.sigmoid(rede(torch.tensor(X_te, dtype=torch.float32)))
y_prev = (p_teste.numpy().ravel() >= 0.5).astype(int)
print(f"Revocacao (ameaca): {recall_score(y_te, y_prev):.3f}   "
      f"Precisao: {precision_score(y_te, y_prev, zero_division=0):.3f}")

**Observe:** as três linhas centrais do laço — `zero_grad()`, `backward()`, `step()` — **são** a retropropagação e o gradiente descendente da Seção 3.2, agora visíveis no código. O que o `fit` do Keras esconde, o PyTorch expõe.

> **✏️ Experimente:** remova as duas linhas `nn.ReLU(),` da rede acima e reexecute. A rede "profunda" colapsa no desempenho de um modelo linear — é a demonstração prática de que, sem a não linearidade entre camadas, empilhar camadas não acrescenta **nada**. (Este é o Exercício 3.2.)

---
## 3.3 Arquiteturas profundas

Um MLP trata a entrada como um vetor **sem estrutura**: se as colunas forem embaralhadas de forma consistente, ele aprende igualmente bem. Mas imagens têm estrutura **espacial** — pixels vizinhos se relacionam — e séries têm estrutura **temporal**. Arquiteturas especializadas embutem essas suposições no próprio desenho da rede, ganhando desempenho e economizando parâmetros.

### 3.3.1 Redes convolucionais (CNN)

A **rede convolucional** é o padrão para imagens. Seu bloco central é a **convolução**: um pequeno filtro (*kernel*) desliza sobre a imagem e calcula, em cada posição, uma soma ponderada local, produzindo um **mapa de ativação**. Para uma imagem $I$ e um filtro $K$,

$$(I * K)(i, j) = \sum_m \sum_n I(i + m,\, j + n)\, K(m, n).$$

Três ideias explicam o seu sucesso. Os **campos receptivos locais**: cada unidade enxerga apenas uma vizinhança, adequada à natureza local das feições de imagem. O **compartilhamento de pesos**: o mesmo filtro é aplicado em toda a imagem, o que confere *invariância à translação* (um objeto é reconhecido onde quer que apareça) e reduz drasticamente o número de parâmetros frente a uma camada densa. E o ***pooling*** (agregação, tipicamente pelo máximo), que reduz a resolução e adiciona robustez a pequenos deslocamentos. Empilhando blocos de convolução, ReLU e *pooling*, a rede constrói uma **hierarquia de feições**: bordas, texturas, partes e, por fim, objetos.

> **🛡️ Contexto de defesa** — As CNNs são a espinha dorsal da **detecção de alvos em imagens** — de satélite, de radar de abertura sintética (SAR) e de vídeo em movimento captado por VANT. O *Project Maven*, do Departamento de Defesa dos EUA, foi precisamente a aplicação de visão computacional profunda a vídeo de sensor, com o objetivo declarado de reduzir a sobrecarga dos analistas na triagem de imagens. O Capítulo 4 desenvolve esse terreno com arquiteturas de detecção como YOLO e Faster R-CNN, que estendem os princípios convolucionais vistos aqui.

Antes de construir uma CNN, vale **ver** uma convolução em ação — a célula abaixo aplica três filtros clássicos, definidos à mão, a uma imagem sintética de sensor:

In [ ]:
# A convolucao em acao: filtros deslizando sobre uma imagem de sensor
from scipy.signal import convolve2d

# Imagem sintetica 64x64: fundo ruidoso + um "alvo" retangular brilhante
rng = np.random.default_rng(SEMENTE)
img = rng.normal(0.3, 0.08, (64, 64))
img[28:36, 20:44] += 0.55            # alvo horizontal
img = np.clip(img, 0, 1)

filtros = {
    "borda vertical":   np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]),
    "borda horizontal": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]),
    "suavizacao":       np.ones((3, 3)) / 9.0,
}

fig, eixos = plt.subplots(1, 4, figsize=(13, 3.4))
eixos[0].imshow(img, cmap="gray")
eixos[0].set_title("imagem de entrada")
for eixo, (nome, K) in zip(eixos[1:], filtros.items()):
    mapa = convolve2d(img, K, mode="same")
    eixo.imshow(np.abs(mapa), cmap="magma")
    eixo.set_title(f"mapa de ativacao:\n{nome}")
for eixo in eixos:
    eixo.axis("off")
plt.tight_layout(); plt.show()

**Observe:** cada filtro **realça um padrão** — bordas verticais, horizontais, regiões suaves. A diferença decisiva da CNN é que ela **não usa filtros desenhados à mão**: os valores dos *kernels* são pesos treináveis, aprendidos por retropropagação para realçar exatamente os padrões que resolvem a tarefa.

A **Listagem 3.3** monta uma CNN compacta para imagens de sensor em tons de cinza ($64 \times 64 \times 1$) — repare, no sumário, na economia de parâmetros frente a uma rede densa equivalente:

In [ ]:
# Listagem 3.3 - CNN compacta em Keras para classificacao de imagens.
# CNN para imagens em tons de cinza de 64x64x1: alvo presente ou ausente.
H, W = 64, 64
cnn = keras.Sequential([
    layers.Input(shape=(H, W, 1)),

    layers.Conv2D(16, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),           # 64x64 -> 32x32

    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),           # 32x32 -> 16x16

    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),           # 16x16 -> 8x8

    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
cnn.compile(optimizer="adam", loss="binary_crossentropy",
            metrics=["accuracy"])
# Note, no sumario, a economia de parametros frente a uma rede densa
# equivalente sobre a imagem achatada (64*64 = 4096 entradas).
cnn.summary()

**Observe:** a primeira camada convolucional tem apenas $16 \times (3 \cdot 3 + 1) = 160$ parâmetros. Uma camada **densa** que recebesse a imagem achatada (4096 entradas) e projetasse em 16 unidades custaria $4096 \times 16 + 16 = 65\,552$ — mais de **400 vezes** mais. É o compartilhamento de pesos em números; e é essa economia que viabiliza a IA **embarcada** do Módulo IV.

### 3.3.2 Redes recorrentes e LSTM

Para *sequências*, a arquitetura natural é a **rede recorrente** (RNN). Ela processa a entrada passo a passo, mantendo um **estado oculto** $\mathbf{h}_t$ que funciona como memória do que já foi visto:

$$\mathbf{h}_t = \varphi\left(\mathbf{W}_x\, \mathbf{x}_t + \mathbf{W}_h\, \mathbf{h}_{t-1} + \mathbf{b}\right).$$

O problema é que, ao retropropagar por sequências longas, o gradiente tende a desaparecer ou a explodir — a RNN simples "esquece" dependências distantes. A **LSTM** (*Long Short-Term Memory*) corrige isso com um *estado de célula* $\mathbf{c}_t$ e três **comportas** que regulam o fluxo de informação — de esquecimento ($\mathbf{f}_t$), de entrada ($\mathbf{i}_t$) e de saída ($\mathbf{o}_t$):

$$\begin{aligned}
\mathbf{f}_t &= \sigma(\mathbf{W}_f\, \mathbf{x}_t + \mathbf{U}_f\, \mathbf{h}_{t-1} + \mathbf{b}_f),\\
\mathbf{i}_t &= \sigma(\mathbf{W}_i\, \mathbf{x}_t + \mathbf{U}_i\, \mathbf{h}_{t-1} + \mathbf{b}_i),\\
\mathbf{o}_t &= \sigma(\mathbf{W}_o\, \mathbf{x}_t + \mathbf{U}_o\, \mathbf{h}_{t-1} + \mathbf{b}_o),\\
\mathbf{c}_t &= \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tanh(\mathbf{W}_c\, \mathbf{x}_t + \mathbf{U}_c\, \mathbf{h}_{t-1} + \mathbf{b}_c),\\
\mathbf{h}_t &= \mathbf{o}_t \odot \tanh(\mathbf{c}_t).
\end{aligned}$$

A comporta de esquecimento decide o que descartar da memória; a de entrada, o que acrescentar; a de saída, o que expor. Essa estrutura **preserva o gradiente** ao longo de horizontes longos. Em defesa, RNNs e LSTMs modelam **séries temporais** de telemetria, previsão de **trajetórias** (faixas de radar, dados AIS) e, historicamente, texto — tema do Capítulo 5.

> **✅ Boa prática** — Para muitas tarefas de sequência, sobretudo em texto, a arquitetura *Transformer* (Capítulo 6) hoje supera as redes recorrentes e paraleliza melhor o treino. A LSTM, porém, permanece forte e **barata** em sequências curtas e em plataformas com restrição de recursos — exatamente o cenário da IA embarcada do Módulo IV. A escolha da arquitetura é, também aqui, uma decisão de engenharia condicionada pelo custo operacional, não apenas pelo estado da arte.

A célula abaixo põe a diferença à prova com o experimento clássico de memória de longo prazo — o ***adding problem*** (Hochreiter e Schmidhuber, 1997), aqui em versão binarizada. Cada sequência traz um canal de **leituras** e um canal de **marcadores** que apontam duas leituras distantes no tempo — uma no início, outra na metade final. A rede deve decidir se a **soma das duas** ultrapassa um limiar, o que a obriga a *carregar a primeira leitura por dezenas de passos* até a decisão (pense em correlacionar dois retornos distantes de uma mesma trilha de radar):

In [ ]:
# RNN simples vs. LSTM no "adding problem" (memoria de longo prazo)
# Cada sequencia tem 2 canais: leituras em [0, 1] e um marcador que
# aponta DUAS posicoes -- uma no inicio, outra na metade final.
# Rotulo: a soma das duas leituras marcadas ultrapassa 1.0?
rng = np.random.default_rng(SEMENTE)
n, T = 3000, 60
leituras = rng.uniform(0, 1, (n, T)).astype("float32")
marcador = np.zeros((n, T), dtype="float32")
pos1 = rng.integers(0, 10, n)         # no inicio da sequencia
pos2 = rng.integers(T // 2, T, n)     # na metade final
marcador[np.arange(n), pos1] = 1.0
marcador[np.arange(n), pos2] = 1.0
X_seq = np.stack([leituras, marcador], axis=-1)
y_seq = ((leituras[np.arange(n), pos1]
          + leituras[np.arange(n), pos2]) > 1.0).astype("float32")

historias = {}
for nome, camada in [("RNN simples", layers.SimpleRNN(16)),
                     ("LSTM", layers.LSTM(16))]:
    keras.utils.set_random_seed(SEMENTE)   # mesma partida para ambas
    modelo = keras.Sequential([
        layers.Input(shape=(T, 2)),
        camada,
        layers.Dense(1, activation="sigmoid"),
    ])
    modelo.compile(optimizer="adam", loss="binary_crossentropy",
                   metrics=["accuracy"])
    h = modelo.fit(X_seq, y_seq, validation_split=0.2, epochs=30,
                   batch_size=64, verbose=0)
    historias[nome] = h.history["val_accuracy"]
    print(f"{nome:12s} | acuracia de validacao final: "
          f"{historias[nome][-1]:.3f}")

plt.figure(figsize=(7, 4.2))
for nome, curva in historias.items():
    plt.plot(range(1, len(curva) + 1), curva, "o-", label=nome)
plt.axhline(0.5, color="gray", ls=":", label="palpite aleatorio")
plt.xlabel("epoca"); plt.ylabel("acuracia de validacao")
plt.title(f"O 'adding problem': carregar uma leitura por ate {T} passos")
plt.legend(); plt.tight_layout(); plt.show()

**Observe:** a RNN simples **avança pouco além do acaso** — o gradiente que liga a decisão (fim da sequência) à informação relevante (a leitura marcada no início) precisa atravessar dezenas de passos e **evanesce no caminho**. A LSTM, com suas comportas, preserva esse gradiente, abre vantagem clara e continuaria melhorando com mais épocas. É a diferença, em miniatura, entre conseguir e não conseguir **correlacionar eventos distantes no tempo**: um contato observado 50 varreduras atrás, uma manobra no início de uma trilha AIS.

---
## 3.4 Regularização, otimização e boas práticas de treinamento

Redes profundas têm capacidade enorme — e, com ela, a tendência a **decorar** os dados de treino em vez de aprender a regularidade que generaliza. Controlar essa tendência é o que separa um modelo de *notebook* de um modelo operacional.

### 3.4.1 Sobreajuste em redes profundas

O sintoma é inequívoco: a perda de **treino** continua a cair enquanto a perda de **validação** estaciona e depois sobe. A rede está memorizando ruído. Por isso, a primeira regra é **monitorar sempre um conjunto de validação** separado — sem ele, o sobreajuste passa despercebido até o fracasso em produção.

### 3.4.2 Técnicas de regularização

Um repertório consolidado combate o sobreajuste:

- **Dropout**: durante o treino, desliga aleatoriamente uma fração $p$ das unidades a cada passo, impedindo que se co-adaptem; na inferência, todas voltam, com a devida escala.
- **Weight decay** (regularização $L_2$): acrescenta $\lambda \lVert \theta \rVert^2$ à perda, favorecendo pesos menores e funções mais suaves.
- **Batch normalization**: normaliza as pré-ativações de cada camada por minilote, estabilizando e acelerando o treino, com efeito regularizador brando.
- **Parada antecipada** (*early stopping*): interrompe o treino quando a perda de validação deixa de melhorar e retém os melhores pesos.
- **Aumento de dados** (*data augmentation*): para imagens, espelhamentos, rotações e recortes aleatórios multiplicam efetivamente o conjunto e ensinam invariâncias — recurso precioso em defesa, onde imagens rotuladas são escassas.

### 3.4.3 Otimização

O SGD puro funciona, mas é lento e sensível. O **momento** acumula uma velocidade que suaviza a trajetória; o **Adam** adapta a taxa de aprendizado por parâmetro a partir de estimativas dos momentos do gradiente e é o otimizador-padrão de fato. Ainda assim, o hiperparâmetro mais importante é a **taxa de aprendizado**: alta demais, o treino diverge; baixa demais, ele se arrasta (como vimos na Seção 3.2). *Agendas* de decaimento e aquecimento ajudam. O tamanho do minilote troca ruído por velocidade.

### 3.4.4 Boas práticas

Fecham o quadro as práticas que, em sistemas críticos, tocam a própria auditabilidade: **normalizar as entradas** (como no Capítulo 2); usar **inicialização** de pesos sensata (He para ReLU, Glorot para $\tanh$); partir sempre de uma ***baseline* forte**; acompanhar as **curvas de validação**; e **fixar sementes** aleatórias para reprodutibilidade — em defesa, um resultado que não se reproduz não se audita. Por fim, manter o modelo **tão pequeno quanto o problema permitir**, antecipando as restrições de energia, memória e latência do Módulo IV.

> **⚠️ Armadilha comum** — **Recorrer ao *deep learning* por reflexo.** Em dados tabulares de volume modesto — a situação frequente em defesa — uma rede profunda costuma *perder* para uma *baseline* clássica bem ajustada (Capítulo 2), além de custar mais para treinar, servir e, sobretudo, auditar. A pergunta correta não mudou: qual é o modelo mais simples que resolve o problema com garantias suficientes? A rede profunda deve ser a resposta a uma necessidade demonstrada, não o ponto de partida.

A célula abaixo produz o sobreajuste **de propósito** — rede grande demais, dados de menos — e mostra o efeito da regularização sobre as curvas:

In [ ]:
# Sobreajuste ao vivo: rede grande demais, dados de menos
# Mesmo problema de triagem da Listagem 3.1, porem com SO 350 exemplos.
rng = np.random.default_rng(SEMENTE)
n_pequeno = 350
Xp = rng.normal(size=(n_pequeno, 12))
logit = (3.0 * Xp[:, 0] * Xp[:, 3] - 1.0 * Xp[:, 5]
         + 1.5 * np.sin(Xp[:, 7]) - 3.0)
yp = (rng.uniform(size=n_pequeno)
      < 1.0 / (1.0 + np.exp(-logit))).astype("int32")

def constroi_mlp(regularizada):
    reg = keras.regularizers.l2(1e-3) if regularizada else None
    blocos = [layers.Input(shape=(12,))]
    for _ in range(2):
        blocos.append(layers.Dense(256, activation="relu",
                                   kernel_regularizer=reg))
        if regularizada:
            blocos.append(layers.Dropout(0.4))
    blocos.append(layers.Dense(1, activation="sigmoid"))
    modelo = keras.Sequential(blocos)
    modelo.compile(optimizer="adam", loss="binary_crossentropy")
    return modelo

fig, eixos = plt.subplots(1, 2, figsize=(11.5, 4.2), sharey=True)
for eixo, reg in [(eixos[0], False), (eixos[1], True)]:
    h = constroi_mlp(reg).fit(Xp, yp, validation_split=0.3,
                              epochs=120, batch_size=32, verbose=0)
    eixo.plot(h.history["loss"], label="perda de treino")
    eixo.plot(h.history["val_loss"], label="perda de validacao")
    eixo.set_title("com dropout + weight decay" if reg
                   else "sem regularizacao")
    eixo.set_xlabel("epoca"); eixo.legend()
eixos[0].set_ylabel("entropia cruzada")
plt.suptitle("O sintoma do sobreajuste: treino cai, validacao sobe")
plt.tight_layout(); plt.show()

**Observe:** à esquerda, o retrato clássico do sobreajuste — a perda de treino despenca rumo a zero (a rede está *decorando* os 350 exemplos) enquanto a de validação **sobe**. À direita, *dropout* e *weight decay* seguram o fenômeno: as curvas caminham juntas. A **parada antecipada** completaria o quadro, interrompendo o treino no ponto em que a validação para de melhorar.

A **Listagem 3.4** consolida essas práticas em um modelo de referência — batch normalization, dropout, weight decay e *callbacks* de validação — que treinaremos na Seção 3.5:

In [ ]:
# Listagem 3.4 - Treino disciplinado: regularizacao e retornos de validacao.
# CNN com regularizacao explicita: batch norm + dropout + weight decay.
modelo = keras.Sequential([
    layers.Input(shape=(64, 64, 1)),

    layers.Conv2D(32, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),     # estabiliza e acelera o treino
    layers.Activation("relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),             # combate a co-adaptacao
    layers.Dense(64, activation="relu",
                 kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])

modelo.compile(optimizer=keras.optimizers.Adam(1e-3),
               loss="binary_crossentropy", metrics=["accuracy"])

# Parada antecipada pela perda de validacao + guarda do melhor modelo.
retornos = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=6,
                                  restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("melhor_modelo.keras",
                                    monitor="val_loss",
                                    save_best_only=True),
]

# O treino em si usa as imagens da Secao 3.5, a seguir:
# historico = modelo.fit(X_tr, y_tr, validation_split=0.2,
#                        epochs=100, batch_size=32, callbacks=retornos)
print(f"Modelo regularizado definido: {modelo.count_params():,} parametros")

---
## 3.5 Aplicação integrada: uma CNN para triagem de imagens de sensor

Reunimos as peças em um problema realista de apoio à decisão. Um fluxo de vídeo de sensor gera **mais quadros do que qualquer equipe de analistas consegue inspecionar**. A tarefa não é substituir o analista, mas **priorizar** a sua atenção: sinalizar os quadros com maior probabilidade de conter um alvo de interesse — o mesmo objetivo declarado do *Project Maven*. A métrica operacional decisiva é a **revocação** da classe-alvo: é tolerável revisar alguns falsos positivos; é grave deixar passar um alvo real.

Como nos capítulos anteriores, trabalhamos com **dados sintéticos** desenhados para reproduzir as dificuldades reais sem reproduzir dados reais: quadros de $64 \times 64$ pixels com fundo de ruído *speckle* (típico de imagens de radar), manchas difusas de *clutter* — e, em parte deles, uma **assinatura alongada de baixo contraste**, em posição e orientação aleatórias: o alvo.

In [ ]:
# Conjunto sintetico de "quadros de sensor": alvo presente ou ausente
from scipy.ndimage import gaussian_filter, rotate
from sklearn.model_selection import train_test_split

def gera_imagens_sensor(n, semente=SEMENTE, lado=64, frac_alvo=0.35):
    '''Quadros lado x lado com fundo tipo speckle + clutter e, em parte
    deles, uma assinatura alongada de baixo contraste (o "alvo").'''
    rng = np.random.default_rng(semente)
    X = np.empty((n, lado, lado, 1), dtype="float32")
    y = (rng.uniform(size=n) < frac_alvo).astype("int32")
    for i in range(n):
        # fundo: ruido speckle suavizado + manchas difusas de clutter
        # (algumas manchas rivalizam com o alvo em brilho local)
        fundo = gaussian_filter(rng.rayleigh(0.22, (lado, lado)), 1.0)
        for _ in range(rng.integers(3, 8)):
            cy, cx = rng.integers(4, lado - 4, 2)
            mancha = np.zeros((lado, lado))
            mancha[cy, cx] = rng.uniform(5, 10)
            fundo += gaussian_filter(mancha, rng.uniform(2.5, 4.0))
        if y[i] == 1:
            # assinatura do alvo: barra alongada de BAIXO contraste
            alvo = np.zeros((lado, lado))
            comp = int(rng.integers(10, 16))
            ay, ax = rng.integers(12, lado - 12, 2)
            alvo[ay - 1:ay + 2,
                 ax - comp // 2:ax + comp // 2] = rng.uniform(0.40, 0.60)
            alvo = rotate(alvo, rng.uniform(0, 180),
                          reshape=False, order=1)
            fundo += gaussian_filter(alvo, 0.8)
        # escala FIXA (nao depende da imagem, para nao vazar o rotulo)
        X[i, :, :, 0] = np.clip(fundo, 0.0, 1.0)
    return X, y

X_img, y_img = gera_imagens_sensor(1800)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_img, y_img, test_size=0.25, stratify=y_img, random_state=SEMENTE)
print(f"treino: {X_tr.shape} | teste: {X_te.shape} | "
      f"fracao de alvos: {y_img.mean():.2f}")

fig, eixos = plt.subplots(2, 6, figsize=(13, 4.6))
for eixo, i in zip(eixos.ravel(), range(12)):
    eixo.imshow(X_img[i, :, :, 0], cmap="gray")
    eixo.set_title("ALVO" if y_img[i] else "fundo", fontsize=10,
                   color="tab:red" if y_img[i] else "gray")
    eixo.axis("off")
plt.suptitle("Amostras do fluxo de quadros do sensor (sinteticas)")
plt.tight_layout(); plt.show()

**Observe:** o alvo **não salta aos olhos** — o contraste é baixo e o *clutter* produz manchas que confundem. É deliberado: uma tarefa de triagem trivial não exigiria aprendizado de máquina.

Em defesa, imagens **rotuladas** são escassas e caras. O **aumento de dados** multiplica efetivamente o conjunto de treino aplicando transformações que preservam o rótulo — espelhamentos, pequenas rotações e *zoom* — e, de quebra, ensina à rede as **invariâncias** do problema: um alvo continua sendo um alvo espelhado ou levemente girado. A célula abaixo mostra o que a rede "verá" durante o treino:

In [ ]:
# Aumento de dados: a mesma imagem, novas variacoes uteis
aumento = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.05),
])

indice_alvo = int(np.where(y_tr == 1)[0][0])
original = X_tr[indice_alvo:indice_alvo + 1]

fig, eixos = plt.subplots(1, 6, figsize=(13, 2.6))
eixos[0].imshow(original[0, :, :, 0], cmap="gray")
eixos[0].set_title("original")
for k, eixo in enumerate(eixos[1:], 1):
    variacao = np.asarray(aumento(original, training=True))
    eixo.imshow(variacao[0, :, :, 0], cmap="gray")
    eixo.set_title(f"variacao {k}")
for eixo in eixos:
    eixo.axis("off")
plt.tight_layout(); plt.show()

A **Listagem 3.5** monta o pipeline completo de triagem: CNN com aumento de dados embutido, treino otimizado para **revocação** com parada antecipada — e, passo inegociável desta obra, o confronto com uma ***baseline* clássica** (floresta aleatória sobre pixels subamostrados). Só a comparação justifica (ou não) o custo adicional da rede profunda em produção.

> **📝 Nota** — Um detalhe de arquitetura, escolhido pela natureza da tarefa: o agregado global final usa o **máximo** (`GlobalMaxPooling2D`), não a média. O alvo ocupa menos de 1% do quadro; na *média* global, sua ativação dilui-se entre milhares de posições de fundo e a rede leva dezenas de épocas para começar a aprender. O *máximo* responde diretamente à pergunta operacional — *"há forte evidência de alvo em **alguma** posição?"* — e acelera o aprendizado. É um exemplo concreto de como a semântica da tarefa deve guiar o desenho da rede.

In [ ]:
# Listagem 3.5 - Pipeline de triagem: CNN com aumento de dados versus
# baseline classica.
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score

# --- 1. Aumento de dados: invariancias uteis, aplicado so no treino ---
aumento = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.05),
])

# --- 2. CNN de triagem (API funcional) ---
entrada = keras.Input(shape=(64, 64, 1))
x = aumento(entrada)
x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = layers.MaxPooling2D()(x)
# Max (nao media) global: o alvo ocupa <1% do quadro, e a pergunta
# operacional e "ha alvo em ALGUM lugar?" -- que o maximo responde.
x = layers.GlobalMaxPooling2D()(x)
x = layers.Dropout(0.4)(x)
saida = layers.Dense(1, activation="sigmoid")(x)
cnn = keras.Model(entrada, saida)

# Otimiza para REVOCACAO: em triagem, o custo de perder um alvo domina.
cnn.compile(optimizer="adam", loss="binary_crossentropy",
            metrics=[keras.metrics.Recall(name="rev")])
parada = keras.callbacks.EarlyStopping(monitor="val_rev", mode="max",
                                       patience=8,
                                       restore_best_weights=True)
cnn.fit(X_tr, y_tr, validation_split=0.2, epochs=40,
        batch_size=32, callbacks=[parada], verbose=0)
p_cnn = (cnn.predict(X_te, verbose=0).ravel() >= 0.5).astype(int)

# --- 3. BASELINE obrigatoria: RandomForest sobre pixels subamostrados ---
Xf_tr = X_tr.reshape(len(X_tr), -1)[:, ::16]
Xf_te = X_te.reshape(len(X_te), -1)[:, ::16]
base = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                              random_state=SEMENTE).fit(Xf_tr, y_tr)
p_base = base.predict(Xf_te)

# --- 4. Comparacao sob a metrica operacional ---
for nome, pred in [("CNN", p_cnn), ("RandomForest", p_base)]:
    r = recall_score(y_te, pred)
    p = precision_score(y_te, pred, zero_division=0)
    print(f"{nome:13s} revocacao={r:.3f}  precisao={p:.3f}")

**Observe:** a CNN explora a estrutura **espacial** dos quadros — exatamente a informação que a floresta, olhando pixels isolados e subamostrados, não consegue usar. Se, no seu ambiente, a vantagem da CNN for pequena, a lição do capítulo se aplica na íntegra: o modelo mais simples, mais barato e mais auditável **ganha a vaga**.

O desfecho operacional não termina no número. A CNN **tria**; o ser humano **decide**. Esse arranjo — o modelo prioriza, o operador julga — é a forma prática do princípio do **controle humano significativo**, que o Capítulo 10 tratará como exigência normativa. Uma triagem que aumenta a revocação sem substituir o julgamento humano amplia a capacidade da equipe; uma automação que o dispensa transfere ao modelo uma responsabilidade que ele não pode carregar.

> **✏️ Experimente:** o limiar de 0,5 usado em `p_cnn` é herdado, não escolhido. Reduza-o para `0.3` e reexecute a comparação: a revocação sobe — e a precisão paga o preço. Em triagem real, esse limiar é uma **decisão de projeto** negociada com o operador (o Exercício 3.5 sistematiza essa análise).

Antes de encerrar, vale olhar **onde** o modelo erra — em triagem, os falsos negativos são os quadros que custam caro:

In [ ]:
# Onde o modelo erra? Leitura operacional da matriz de confusao
from sklearn.metrics import confusion_matrix

vn, fp, fn, vp = confusion_matrix(y_te, p_cnn).ravel()
print("                      previsto: fundo | previsto: ALVO")
print(f"real: fundo               {vn:5d}        |     {fp:5d}  <- alarmes falsos")
print(f"real: ALVO                {fn:5d}        |     {vp:5d}")
print(f"\n{fn} alvos passariam SEM revisao humana (falsos negativos)")

indices_fn = np.where((y_te == 1) & (p_cnn == 0))[0]
if len(indices_fn):
    k = min(6, len(indices_fn))
    fig, eixos = plt.subplots(1, k, figsize=(2.2 * k, 2.6))
    for eixo, i in zip(np.atleast_1d(eixos), indices_fn[:k]):
        eixo.imshow(X_te[i, :, :, 0], cmap="gray")
        eixo.axis("off")
    plt.suptitle("Falsos negativos: os quadros que a triagem deixou passar")
    plt.tight_layout(); plt.show()

---
## 3.6 O caminho à frente

Com o perceptron, a retropropagação e as arquiteturas convolucional e recorrente, temos o **motor do aprendizado profundo** — e, com ele, fechamos o Módulo I. O Módulo II põe esse motor a serviço da *percepção*. O **Capítulo 4** desenvolve a visão computacional para defesa, levando adiante as CNNs vistas aqui até arquiteturas de detecção de alvos (YOLO, Faster R-CNN) e o tratamento de imagens SAR e de VANT. O **Capítulo 5** trata do processamento de linguagem natural para inteligência. E o **Capítulo 6** chega aos modelos de linguagem de grande escala e à arquitetura *Transformer*, que hoje supera as redes recorrentes em quase toda tarefa de sequência.

A disciplina que atravessou os três capítulos do Módulo I — comparar contra uma *baseline* honesta, escolher métricas ao custo operacional do erro, manter o humano no centro da decisão — não se dissolve à medida que os modelos ganham capacidade. Torna-se, a cada capítulo, mais indispensável.

## 📋 Resumo do capítulo

- O **perceptron** classifica por um hiperplano e só resolve problemas linearmente separáveis — o **XOR** expõe seu limite. O **perceptron multicamadas**, ao intercalar não linearidades entre camadas, aproxima qualquer função contínua (aproximação universal).

- O treino minimiza a perda por **gradiente descendente**; a **retropropagação** calcula o gradiente pela regra da cadeia, propagando o erro para trás. A não linearidade é essencial, e a **ReLU** substituiu a sigmoide nas camadas ocultas para evitar o *gradiente evanescente*.

- As **redes convolucionais** exploram a estrutura espacial das imagens por campos receptivos locais, compartilhamento de pesos e *pooling*, construindo feições hierárquicas — são a base da visão computacional para defesa.

- As **redes recorrentes** e, sobretudo, as **LSTM** tratam sequências mantendo memória; as comportas da LSTM preservam o gradiente em horizontes longos, viabilizando séries temporais e trajetórias.

- Contra o sobreajuste, combinam-se **dropout**, **weight decay**, **batch normalization**, **parada antecipada** e **aumento de dados**; o **Adam** é o otimizador-padrão, e a **taxa de aprendizado**, o hiperparâmetro mais crítico.

- `TensorFlow/Keras` (idioma **declarativo**) e `PyTorch` (idioma **imperativo**) são os dois padrões de mercado — e esta obra emprega os dois.

- O *deep learning* **amplia** o repertório, mas não **substitui** a *baseline* clássica: uma rede profunda só se justifica quando a supera com folga, ao custo de menor auditabilidade — e a decisão final permanece humana.

## ⚠️ Armadilhas comuns

1. **Adotar *deep learning* por reflexo.** Em dados tabulares de volume modesto, uma *baseline* clássica bem ajustada costuma vencer a rede profunda, a custo menor e com maior auditabilidade. Exija que a rede *prove* sua vantagem.

2. **Não normalizar as entradas.** Redes treinam mal com atributos em escalas díspares. Padronize (e faça-o ajustando as estatísticas apenas no treino, para não vazar informação do teste).

3. **Empilhar camadas sem ativação não linear.** Uma pilha de camadas puramente lineares colapsa em um único mapa linear — toda a expressividade extra se perde. A não linearidade entre camadas é o que dá poder à rede.

4. **Usar sigmoide nas camadas ocultas de uma rede profunda.** A saturação provoca gradiente evanescente e trava o aprendizado das primeiras camadas. Prefira ReLU (ou variantes) nas ocultas; reserve a sigmoide para a saída binária.

5. **Treinar sem conjunto de validação.** Sem acompanhar a perda de validação, o sobreajuste avança em silêncio e só se revela — tarde — em produção. Monitore-a e use parada antecipada.

6. **Tratar a rede como caixa-preta em sistema crítico.** Sem sementes fixas, versionamento e rastreabilidade, o resultado não se reproduz nem se audita — e, sem controle humano na decisão, a automação assume um risco que não lhe cabe (Capítulo 10).

---
## 🧭 Exercícios

Classificação: **Essencial** (fixação) · **Tático** (aplicação) · **Estratégico** (extensão criativa). Os exercícios de código têm células preparadas abaixo; os dissertativos podem ser respondidos em células de texto no próprio notebook.

### Essencial

**Exercício 3.1** — Execute a célula abaixo (Listagem 3.1 + *baseline*) e compare a revocação e a precisão do MLP às de uma **regressão logística** (Capítulo 2) sobre os *mesmos* dados. Em seguida, **varie o número de unidades** das camadas ocultas em $\{16, 64, 256\}$ e relate, em poucas linhas, o efeito sobre as duas métricas e sobre os sinais de sobreajuste.

In [ ]:
# Exercicio 3.1 - MLP vs. regressao logistica; efeito da largura da rede
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score

rng = np.random.default_rng(SEMENTE)
n, d = 4000, 12
X_tab = rng.normal(size=(n, d))
logit = (3.0 * X_tab[:, 0] * X_tab[:, 3] - 1.0 * X_tab[:, 5]
         + 1.5 * np.sin(X_tab[:, 7]) - 3.0)
y_tab = (rng.uniform(size=n) < 1.0 / (1.0 + np.exp(-logit))).astype("int32")

Xtab_tr, Xtab_te, ytab_tr, ytab_te = train_test_split(
    X_tab, y_tab, stratify=y_tab, test_size=0.25, random_state=SEMENTE)
scaler = StandardScaler().fit(Xtab_tr)
Xtab_tr, Xtab_te = scaler.transform(Xtab_tr), scaler.transform(Xtab_te)

# Baseline do Capitulo 2: regressao logistica (modelo LINEAR)
logistica = LogisticRegression(class_weight="balanced",
                               max_iter=1000).fit(Xtab_tr, ytab_tr)
p_log = logistica.predict(Xtab_te)

# MLP com largura ajustavel
unidades = 64            # <--- ALTERE AQUI: teste 16, 64 e 256
mlp = keras.Sequential([
    layers.Input(shape=(d,)),
    layers.Dense(unidades, activation="relu"),
    layers.Dense(unidades // 2, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
mlp.compile(optimizer=keras.optimizers.Adam(1e-3),
            loss="binary_crossentropy")
peso = {0: 1.0, 1: float((ytab_tr == 0).sum() / max((ytab_tr == 1).sum(), 1))}
h = mlp.fit(Xtab_tr, ytab_tr, validation_split=0.2, epochs=30,
            batch_size=64, class_weight=peso, verbose=0)
p_mlp = (mlp.predict(Xtab_te, verbose=0).ravel() >= 0.5).astype(int)

print(f"{'modelo':22s} revocacao | precisao")
for nome, pred in [("regressao logistica", p_log),
                   (f"MLP ({unidades} unidades)", p_mlp)]:
    print(f"{nome:22s}   {recall_score(ytab_te, pred):.3f}   |  "
          f"{precision_score(ytab_te, pred, zero_division=0):.3f}")
print(f"perda de treino final: {h.history['loss'][-1]:.3f} | "
      f"validacao: {h.history['val_loss'][-1]:.3f}")

# Sua resposta (poucas linhas):
# 1) MLP vs. logistica neste problema NAO linear: ...
# 2) Efeito de 16 -> 64 -> 256 unidades sobre as metricas: ...
# 3) Sinais de sobreajuste observados (perda treino vs. validacao): ...

**Exercício 3.2** — Na célula abaixo (cópia da Listagem 3.2), **remova** as ativações `nn.ReLU()`, tornando a rede puramente linear. Reexecute e explique, em uma frase, por que o desempenho colapsa ao de um modelo linear — relacionando o resultado ao papel da **não linearidade** entre camadas.

In [ ]:
# Exercicio 3.2 - o papel da nao linearidade entre camadas
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, precision_score

rng = np.random.default_rng(SEMENTE)
n, d = 4000, 12
X_tab = rng.normal(size=(n, d))
logit = (3.0 * X_tab[:, 0] * X_tab[:, 3] - 1.0 * X_tab[:, 5]
         + 1.5 * np.sin(X_tab[:, 7]) - 3.0)
y_tab = (rng.uniform(size=n) < 1.0 / (1.0 + np.exp(-logit))).astype("int32")
Xtab_tr, Xtab_te, ytab_tr, ytab_te = train_test_split(
    X_tab, y_tab, stratify=y_tab, test_size=0.25, random_state=SEMENTE)
scaler = StandardScaler().fit(Xtab_tr)
Xtab_tr, Xtab_te = scaler.transform(Xtab_tr), scaler.transform(Xtab_te)

Xt = torch.tensor(Xtab_tr, dtype=torch.float32)
yt = torch.tensor(ytab_tr, dtype=torch.float32).unsqueeze(1)

torch.manual_seed(SEMENTE)
rede = nn.Sequential(
    nn.Linear(d, 64),
    nn.ReLU(),               # <--- REMOVA esta linha ...
    nn.Linear(64, 32),
    nn.ReLU(),               # <--- ... e esta, e reexecute
    nn.Linear(32, 1),
)

peso_pos = float((ytab_tr == 0).sum() / max((ytab_tr == 1).sum(), 1))
criterio = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(peso_pos))
otimizador = torch.optim.Adam(rede.parameters(), lr=1e-3)
for epoca in range(300):
    otimizador.zero_grad()
    perda = criterio(rede(Xt), yt)
    perda.backward()
    otimizador.step()

with torch.no_grad():
    p = torch.sigmoid(rede(torch.tensor(Xtab_te, dtype=torch.float32)))
y_prev = (p.numpy().ravel() >= 0.5).astype(int)
print(f"Revocacao: {recall_score(ytab_te, y_prev):.3f} | "
      f"Precisao: {precision_score(ytab_te, y_prev, zero_division=0):.3f}")

# Sua resposta (1 frase):
# Sem as ativacoes, a composicao de camadas lineares equivale a ...

**Exercício 3.3** — Execute `cnn.summary()` na célula abaixo (cópia da Listagem 3.3) e anote o **total de parâmetros**. Calcule quantos parâmetros teria a *primeira* camada de uma rede **densa** que recebesse a imagem achatada ($64 \times 64 = 4096$ entradas) e projetasse em 16 unidades. Compare os números e explique, em duas linhas, a economia trazida pelo **compartilhamento de pesos**.

In [ ]:
# Exercicio 3.3 - economia de parametros: convolucao vs. camada densa
# (usa o nome cnn_ex para NAO sobrescrever a CNN treinada da Listagem 3.5)
H, W = 64, 64
cnn_ex = keras.Sequential([
    layers.Input(shape=(H, W, 1)),
    layers.Conv2D(16, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
cnn_ex.summary()

# Complete o calculo da camada densa equivalente:
entradas = 64 * 64
unidades = 16
parametros_densa = 0     # <--- ALTERE AQUI: entradas * unidades + unidades
parametros_conv1 = 16 * (3 * 3 * 1 + 1)
print(f"\n1a camada convolucional: {parametros_conv1} parametros")
print(f"1a camada densa (4096 -> 16): {parametros_densa} parametros")

# Sua resposta (2 linhas):
# A economia vem do compartilhamento de pesos porque ...

### Tático

**Exercício 3.4** — Usando a arquitetura da Listagem 3.4, a célula abaixo treina o modelo sobre as imagens de sensor **com** e **sem** *dropout* e *batch normalization*. Execute-a, registre as curvas de perda de treino e de validação nos dois casos e identifique, em cada um, a **época em que o sobreajuste começa a se manifestar**. *(Requer a função `gera_imagens_sensor` da Seção 3.5 já executada.)*

In [ ]:
# Exercicio 3.4 - efeito de dropout + batch normalization nas curvas
# (requer a funcao gera_imagens_sensor da Secao 3.5 ja executada)
X_ex, y_ex = gera_imagens_sensor(900, semente=1)

def constroi_cnn(com_regularizacao):
    blocos = [layers.Input(shape=(64, 64, 1))]
    for filtros in [32, 64]:
        blocos.append(layers.Conv2D(filtros, 3, padding="same",
                                    use_bias=not com_regularizacao))
        if com_regularizacao:
            blocos.append(layers.BatchNormalization())
        blocos += [layers.Activation("relu"), layers.MaxPooling2D()]
    blocos.append(layers.GlobalMaxPooling2D())
    if com_regularizacao:
        blocos.append(layers.Dropout(0.4))
    blocos += [layers.Dense(64, activation="relu"),
               layers.Dense(1, activation="sigmoid")]
    modelo = keras.Sequential(blocos)
    modelo.compile(optimizer=keras.optimizers.Adam(1e-3),
                   loss="binary_crossentropy")
    return modelo

fig, eixos = plt.subplots(1, 2, figsize=(11.5, 4.2), sharey=True)
for eixo, reg in [(eixos[0], False), (eixos[1], True)]:
    h = constroi_cnn(reg).fit(X_ex, y_ex, validation_split=0.25,
                              epochs=25, batch_size=32, verbose=0)
    eixo.plot(h.history["loss"], label="treino")
    eixo.plot(h.history["val_loss"], label="validacao")
    eixo.set_title("com dropout + batch norm" if reg
                   else "sem regularizacao")
    eixo.set_xlabel("epoca"); eixo.legend()
eixos[0].set_ylabel("entropia cruzada")
plt.tight_layout(); plt.show()

# Sua resposta (poucas linhas):
# 1) Epoca em que o sobreajuste comeca SEM regularizacao: ...
# 2) E COM regularizacao (ou por que ele nao se manifesta): ...

**Exercício 3.5** — Estenda a Listagem 3.5 calculando a curva *precision–recall* da CNN (use `precision_recall_curve` sobre as probabilidades previstas). Escolha um limiar que garanta **revocação da classe-alvo $\geq 0{,}95$** e relate a precisão resultante — repetindo, agora para uma rede, a análise de custo do Exercício 2.5. *(Requer a CNN da Listagem 3.5 treinada.)*

In [ ]:
# Exercicio 3.5 - curva precision-recall da CNN e limiar operacional
# (requer a CNN da Listagem 3.5 treinada)
from sklearn.metrics import precision_recall_curve

probas = cnn.predict(X_te, verbose=0).ravel()
precisao, revocacao, limiares = precision_recall_curve(y_te, probas)

plt.figure(figsize=(7, 4.5))
plt.plot(revocacao, precisao, lw=2)
plt.axvline(0.95, color="r", ls="--", lw=1.2,
            label="revocacao alvo >= 0.95")
plt.xlabel("Revocacao da classe-alvo (recall)")
plt.ylabel("Precisao")
plt.title("Curva precision-recall da CNN de triagem")
plt.legend(); plt.tight_layout(); plt.show()

# Menor limiar cuja revocacao ainda e >= 0.95 e o preco em precisao
mascara = revocacao[:-1] >= 0.95
if mascara.any():
    indice = np.where(mascara)[0][-1]
    print(f"Limiar sugerido:        {limiares[indice]:.3f}")
    print(f"Revocacao nesse ponto:  {revocacao[indice]:.3f}")
    print(f"Precisao nesse ponto:   {precisao[indice]:.3f}")
else:
    print("Nenhum limiar atinge revocacao >= 0.95 -- discuta por que.")

# Sua resposta (poucas linhas):
# 1) Precisao ao garantir revocacao >= 0.95: ...
# 2) Custo operacional (falsos positivos que o operador revisara): ...

**Exercício 3.6** *(dissertativo)* — Considere o problema de **previsão de trajetória** a partir de uma faixa de radar (série temporal de posições). Decida entre uma **RNN simples**, uma **LSTM** e um **MLP sobre janelas fixas**, justifique a escolha e explique, em termos de **gradiente evanescente**, o risco de tentar capturar dependências longas com uma RNN simples.

*Responda em uma célula de texto abaixo.*

### Estratégico

**Exercício 3.7** — Em um breve texto técnico (até duas páginas), discuta em que circunstâncias a adoção de um modelo profundo justifica *substituir* uma *baseline* clássica em um sistema crítico de defesa, dado o custo em auditabilidade. Proponha **critérios objetivos de decisão** — mensuráveis — que uma equipe poderia adotar antes de levar uma rede profunda à produção.

**Exercício 3.8** — Projete (em diagrama ou pseudocódigo) um **pipeline de triagem de imagens** para apoio ao analista, no espírito do *Project Maven*. Especifique os pontos de **controle humano** na decisão, a métrica operacional adotada e as salvaguardas contra o **viés de automação** (a tendência a confiar acriticamente na saída do modelo). Explique por que a revocação é priorizada.

**Exercício 3.9** — Escolha um conjunto público de imagens e conduza um **estudo comparativo completo** entre uma CNN compacta com aumento de dados e uma *baseline* clássica, redigindo um miniartigo de até três páginas que documente: a definição do problema, as escolhas de aumento de dados, as métricas com sua variância e uma recomendação fundamentada — incluindo, de forma explícita, os limites de generalização para imagens operacionais reais e os riscos de **deslocamento de distribuição** (*data shift*).

---

*Fim do Capítulo 3 — encerramos o Módulo I. No Capítulo 4, a visão computacional para defesa: das CNNs às arquiteturas de detecção de alvos.*